# 03 — Tools: Teacher's Assistant (Agent-as-Tool)

**Module 2 of the workshop — the centerpiece.** One orchestrator agent routes to specialist agents that are themselves wrapped as tools. This pattern reappears as the foundation of Module 3 (multi-agent patterns).


## Problem

An LLM alone can't read a file, call an API, or run code — and mixing every capability into one agent's system prompt fights itself once you have several distinct specialties (math, CS, language, general). Tools are the mechanism for acting beyond text; **agent-as-tool** is the mechanism for keeping each specialty's own agent simple while still exposing it to one orchestrator.


## Concept

Four mechanisms, one interface — the agent doesn't care which kind of tool it's calling:

```
                 TOOLS
                   │
       ┌───────────┼────────────┐
       │           │            │
     Custom      Vended        MCP
      Tools       Tools        Tools
       │           │            │
       └───────────┼────────────┘
                   │
             Agent-as-tool
```

The minimum code for this pattern is the `@tool` decorator on a plain function — `math_assistant` below is the clearest example: an entire `Agent` (with its own system prompt and tools) wrapped as a single callable tool for another agent. **What happens internally:** the orchestrator's model decides "this needs `math_assistant`" → emits a structured tool call → Strands executes the wrapped function (which itself runs a full agent loop) → the specialist's answer returns as the tool's result → orchestrator continues.


## Architecture

```
"What is the derivative of x^3 + 2x?"
              │
              ▼
       ┌──────────────┐
       │ teacher_agent │   (orchestrator, callback_handler=None)
       └──────┬───────┘
              │ routes to ONE tool based on query
              │
   ┌──────────┼───────────┬─────────────────┐
   ▼          ▼           ▼                 ▼
math_       computer_   language_        general_
assistant   science_    assistant        assistant
   │        assistant       │                │
   │ @tool     │ @tool       │ @tool           │ @tool
   ▼           ▼             ▼                 ▼
 Agent      Agent          Agent            Agent
 (calculator)(editor,      (http_request)   (no tools)
             file_read/write)

Each specialist is a FULL Agent with its own system_prompt and tools —
wrapped in a @tool function so the orchestrator can call it like any
other tool. The orchestrator never sees the specialist's internal loop,
only its final string answer.
```


## Step 1 — Resolve the model

Same model, reused across the orchestrator and every specialist agent.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from model_provider import get_model
from strands import Agent, tool
from strands_tools import calculator, editor, file_read, file_write, http_request

# NOTE: strands_tools.shell / python_repl import POSIX-only `pty`/`fcntl`,
# unavailable on Windows. Dropped from computer_science_assistant below.

model = get_model()
print(f"Using: {type(model).__name__}")


Using: OllamaModel


## Step 2 — Define each specialist as a `@tool`-wrapped agent

This is the pattern: a plain function, decorated with `@tool`, that builds and runs a *complete* `Agent` internally, then returns its string output. Four specialists, four different toolsets.


In [2]:
@tool
def math_assistant(query: str) -> str:
    """Solve math problems, showing steps."""
    agent = Agent(
        model=model,
        system_prompt="You are a math tutor. Solve problems step by step.",
        tools=[calculator],
    )
    return str(agent(query))


@tool
def computer_science_assistant(query: str) -> str:
    """Answer CS/programming questions, can run code."""
    agent = Agent(
        model=model,
        system_prompt="You are a CS tutor. Explain concepts clearly with examples.",
        tools=[editor, file_read, file_write],
    )
    return str(agent(query))


@tool
def language_assistant(query: str) -> str:
    """Handle translation / language questions."""
    agent = Agent(
        model=model,
        system_prompt="You are a language tutor and translator.",
        tools=[http_request],
    )
    return str(agent(query))


@tool
def general_assistant(query: str) -> str:
    """Fallback for anything else."""
    agent = Agent(model=model, system_prompt="You are a helpful general-purpose tutor.")
    return str(agent(query))


## Step 3 — Build the orchestrator

`teacher_agent` doesn't do math, CS, translation, or anything else itself — it only routes to the single most relevant specialist tool and returns that answer. `callback_handler=None` keeps the orchestrator's own reasoning quiet so only the specialist's final answer prints.


In [3]:
TEACHER_SYSTEM_PROMPT = """You are a Teacher's Assistant. Route each query to the
single most relevant tool (math_assistant, computer_science_assistant,
language_assistant, general_assistant) and return its answer."""

teacher_agent = Agent(
    model=model,
    system_prompt=TEACHER_SYSTEM_PROMPT,
    callback_handler=None,
    tools=[math_assistant, language_assistant, computer_science_assistant, general_assistant],
)


## Step 4 — Run it

Watch the orchestrator pick `math_assistant` specifically, not one of the other three — that routing decision is the whole point of this pattern.


In [4]:
print(teacher_agent("What is the derivative of x^3 + 2x?"))

DEPRECATION WARNING: calculator is deprecated. This warning becomes an error log in v0.9.0. To achieve similar functionality, use the shell tool vended by strands-agents (from strands.vended_tools import shell). This does change the security boundary: calculator only ever evaluated an expression checked against an AST allowlist, while shell executes arbitrary commands, so review it against your threat model before switching.



Tool #1: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭─────────────────┬───────────────────────────╮                                                                │
│  │ Operation       │ Calculate 1-th Derivative │                                                                │
│  │ Input           │ x**3 + 2*x                │                                                                │
│  │ Result          │ 3*x**2 + 2                │                                                                │
│  │ With respect to │ x                         │                                                                │
│  ╰─────────────────┴───────────────────────────╯                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The derivative of **x³ + 2x** with respect to **x** is:

## **3x² + 2**

### Step-by-step Solution:

To find the derivative, we apply the **power rule** for differentiation:  
$\frac{d}{dx}(x^n) = nx^{n-1}$

We differentiate each term separately:

1. **For $x^3$**:
   - Apply the power rule with $n = 3$
   - $\frac{d}{dx}(x^3) = 3x^{3-1} = 3x^2$

2. **For $2x$**:
   - The derivative of $x$ is 1 (since $x^1$, so $1 \cdot x^{1-1} = 1 \cdot x^0 = 1$)
   - $\frac{d}{dx}(2x) = 2 \cdot 1 = 2$

3. **Combine the results**:
   - Since differentiation is linear: $\frac{d}{dx}[f(x) + g(x)] = f'(x) + g'(x)$
   - $\frac{d}{dx}(x^3 + 2x) = 3x^2 + 2$

### Visual Representation:
```
f(x)        = x³ + 2x
f'(x)       = 3x² + 2
```

The derivative tells us the rate of change of the function at any point $x$.The derivative of **x³ + 2x** with respect to **x** is:

## **3x² + 2**

### Step-by-step Solution:

To find the derivative, we apply the **power rule** for differentiation:  
$\frac{d}{dx}(x^n) = nx

## Failure mode to know about

A tool with an ambiguous docstring gets called at the wrong time, or not at all — the model routes on the tool's *description*, not its implementation. If `math_assistant`'s docstring were vague ("helps with stuff"), the orchestrator might route a math question to `general_assistant` instead. A bad tool description is a bug, even if the function itself is correct.
